In [ ]:
class Robot:
    """Represents a single robot in the warehouse"""

    def __init__(self, robot_id, grid, start_pos, goal_pos, color='blue'):
        """Initialize robot with grid reference

        Args:
            robot_id (int): Unique identifier for robot
            grid (GridEnvironment): The grid this robot moves in
            start_pos (tuple): Starting (x, y) position
            goal_pos (tuple): Goal (x, y) position
            color (str): Robot color for visualization
        """
        # --- Static (fixed for lifetime of problem) ---
        self.robot_id  = robot_id
        self.grid      = grid
        self.start_pos = start_pos       # fixed start, needed for reset()
        self.goal_pos  = goal_pos
        self.color     = color

        # --- Dynamic (updated during simulation) ---
        self.current_pos  = start_pos   # tracks live position
        self.path         = []          # planned path: list of (x, y)
        self.path_index   = 0           # current position in path
        self.arrival_time = -1          # timestep when goal first reached (-1 = not yet)
        self.steps_taken  = 0          # moves made (non-wait actions)
        self.waits        = 0          # wait actions taken
        self.status       = 'idle'     # idle | moving | waiting | done

    # ------------------------------------------------------------------ #
    #  Getters                                                             #
    # ------------------------------------------------------------------ #

    def get_current_position(self):
        """Get current position

        Returns:
            tuple: Current (x, y) position
        """
        return self.current_pos

    def get_start_position(self):
        """Get start position

        Returns:
            tuple: Start (x, y) position
        """
        return self.start_pos

    def get_goal_position(self):
        """Get goal position

        Returns:
            tuple: Goal (x, y) position
        """
        return self.goal_pos

    def get_grid(self):
        """Get grid reference

        Returns:
            GridEnvironment: The grid this robot moves in
        """
        return self.grid

    def get_path(self):
        """Get stored path

        Returns:
            List[tuple]: Planned path as list of (x, y) coordinates
        """
        return self.path

    def get_path_length(self):
        """Total time steps in planned path (including waits).

        Returns:
            int: Length of path minus 1 (number of moves)
        """
        return len(self.path) - 1 if self.path else 0

    def get_individual_flowtime(self):
        """Time steps until robot first reached goal.

        Returns:
            int: Arrival timestep, or -1 if not yet reached
        """
        return self.arrival_time

    def get_position_at_time(self, time_step):
        """Get robot's position at a specific time step (for collision detection)

        Args:
            time_step (int): Time step

        Returns:
            tuple: (x, y) position at that time, or last position if past path length
        """
        if time_step < len(self.path):
            return self.path[time_step]
        else:
            # Robot waits at goal after reaching it
            return self.path[-1] if self.path else self.current_pos

    def get_neighbors_from_grid(self):
        """Get valid neighboring positions using grid method

        Returns:
            List[tuple]: List of valid neighbor (x, y) coordinates
        """
        x, y = self.current_pos
        return self.grid.get_neighbors(x, y)

    def is_at_goal(self):
        """Check if robot reached goal

        Returns:
            bool: True if at goal position
        """
        return self.current_pos == self.goal_pos

    def is_done(self):
        """Check if robot has completed its task

        Returns:
            bool: True if status is 'done'
        """
        return self.status == 'done'

    # ------------------------------------------------------------------ #
    #  Path management                                                     #
    # ------------------------------------------------------------------ #

    def set_path(self, path):
        """Store planned path for robot

        Args:
            path (List[tuple]): Path as list of (x, y) coordinates
        """
        self.path       = path
        self.path_index = 0

    # ------------------------------------------------------------------ #
    #  State update helpers (called by Problem.apply_action)              #
    # ------------------------------------------------------------------ #

    def move_to(self, position, timestep):
        """Move robot to new position and update stats.

        Args:
            position (tuple): New (x, y) position
            timestep (int)  : Current simulation timestep
        """
        if not self.grid.is_walkable(position[0], position[1]):
            raise ValueError(f'Cannot move to non-walkable position {position}')

        if position == self.current_pos:
            self.waits  += 1
            self.status  = 'waiting'
        else:
            self.steps_taken += 1
            self.status       = 'moving'

        self.current_pos = position
        self.path_index += 1

        if position == self.goal_pos and self.arrival_time == -1:
            self.arrival_time = timestep
            self.status       = 'done'

    def reset(self):
        """Reset all dynamic attributes back to initial values (for re-planning)."""
        self.current_pos  = self.start_pos
        self.path         = []
        self.path_index   = 0
        self.arrival_time = -1
        self.steps_taken  = 0
        self.waits        = 0
        self.status       = 'idle'

    # ------------------------------------------------------------------ #
    #  Serialisation (for state dict)                                     #
    # ------------------------------------------------------------------ #

    def to_dict(self):
        """Produce the robot entry that lives inside the state['robots'] list.

        Returns:
            dict: Robot state dictionary
        """
        return {
            'id'            : self.robot_id,
            'start_position': self.start_pos,
            'goal_position' : self.goal_pos,
            'color'         : self.color,
            'at_goal'       : self.is_at_goal(),
            'status'        : self.status,
            'arrival_time'  : self.arrival_time,
            'steps_taken'   : self.steps_taken,
            'waits'         : self.waits,
            'grid'          : self.grid,
        }

    def __repr__(self):
        return (f"Robot(id={self.robot_id}, pos={self.current_pos}, "
                f"goal={self.goal_pos}, status={self.status})")

print("Robot class defined")

"""
 puts the robot back to its initial state as if it was never used.
 Useful when you want to re-run a different algorithm on the same robot without creating a new object:

 to_dict() — converts the robot object into a plain dictionary
 so it can be stored inside the state dict (state['robots']). 
 The whole system passes states around as dictionaries (not Robot objects), 
 so this is the bridge between the two:
"""